In [ ]:
# SPDX-FileCopyrightText: 2024 Dan J. Bower <dbower@eaps.ethz.ch>
#
# SPDX-License-Identifier: GPL-3.0-or-later

import logging

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    PurePhase,
    ThermodynamicState,
    debug_logger,
)

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

# Gas Mixing

This notebook is available at `notebooks/gas_mixing.ipynb` and is easiest to obtain by downloading the source code.

In gas mixing experiments, a predefined gaseous mixture is injected into a chamber and allowed to equilibrate thermodynamically under controlled temperature-pressure conditions. In this example, we consider a gas mixture that equilibrates in the presence of solid carbon (graphite).

In [ ]:
# Define allowable gas species at equilibrium
H2_g = ChemicalSpecies.create_gas("H2")
N2_g = ChemicalSpecies.create_gas("N2")
CH4_g = ChemicalSpecies.create_gas("CH4")
CHN_g = ChemicalSpecies.create_gas("CHN")
H_g = ChemicalSpecies.create_gas("H")
gas_species = (H2_g, N2_g, CH4_g, CHN_g, H_g)

# Enforce graphite as a condensate (a pure phase in this model), so
# solve_for_stability=False
graphite = PurePhase.from_species("C", state="s", solve_for_stability=False)
condensate_phases = (graphite,)

# Set the thermodynamic state of the system
state = ThermodynamicState.from_species(
    gas_species, pressure=1, temperature=1773.15, melt_fraction=0, condensates=condensate_phases
)

# Define the mole fractions of input gases
mole_fractions = {"H2": 0.5, "N2": 0.5}

# Now define the model
model = EquilibriumModel.from_state(state, mass_constraints=mole_fractions, mass_units="moles")

# Solve with an automatic initial guess and default solver settings
output = model.solve_with_default()

# Optionally print solver stats to the logger
output.solver_stats_to_logger()

# The abundance of C cannot be solved for, so the output result for the number density of graphite
# should be ignored.
output.quick_look()

# Optionally dump the output to Excel (uncomment the line below)
# output.to_excel("gas_mixing")